# Data Cleaning 04 -- IBES Revenue

## Input
`Data/Data_Collection/Initial/04_LSEG_IBES/ibes_revenue.parquet` (1,150,491 rows across 12,397 IBES tickers)

## Purpose
Cleans monthly stock-level analyst revenue estimate data from WRDS IBES (statsum_xepsus). The raw file contains the entire IBES US coverage universe. This notebook filters to the top-100 S&P 500 universe, then investigates NaN patterns in depth -- particularly the fiscal year rollover effect on revision columns and the structural sparsity of quarterly revision data.

## Stage 0: Load, Filter to Universe, Inspect
- Raw file filtered from 12,397 tickers to the 231 tickers mapping to the 227 master PERMNOs using `ibes_permno_link_clean.parquet`
- Trimmed to 2004-01-01 onwards
- Reduced from 1,150,491 to 48,768 rows
- Basic shape, date range, ticker counts, and per-ticker coverage statistics reported

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts with flags for columns above 10% and 30%
- Per-row NaN distribution

## Stage 2: Value Range & Quality Checks
- **Revenue estimate levels:** range checks on `rev_mean_fy1`, flags negative and zero values
- **Analyst count:** distribution of `rev_numest`, fraction of single-analyst observations
- **Revision columns:** distribution and percentile analysis of `rev_revision_1m` and `rev_revision_3m`, counts extreme values
- **Dispersion:** distribution and percentile analysis of `rev_dispersion`
- **Surprise:** distribution and NaN rate of `rev_surprise` and `rev_surprise_pct`
- **Consistency checks:** verifies revenue estimates are internally consistent (high >= low, mean within bounds)
- **NaN pattern -- dispersion:** confirms NaN occurs exclusively when analyst count = 1
- **NaN pattern -- revision:** investigates first-month-per-ticker NaN
- **Coverage per ticker:** months of data per ticker, identifies tickers with fewer than 24 months
- **Duplicate (ticker, date) check**
- **Date frequency:** checks whether dates are month-end (they are not -- mid-month IBES statistical period dates)

## Stage 2b: Deep NaN Investigation
A detailed investigation of the ~8.4% NaN cluster and higher-NaN columns:
- **~8.4% NaN cluster** (`rev_revision_1m`, `rev_numest_chg`, `rev_fy2_revision_1m`, `rev_eps_divergence`): confirmed that all four columns are NaN on exactly the same rows. Analysis by month reveals 64.5% of NaN concentrates in February -- the fiscal year rollover month for December-FY-end companies. ~21 NaN per ticker across 252 months matches exactly 1 rollover per year.
- **`rev_revision_3m` (25.2% NaN):** the 3-month lookback extends the FY rollover effect across Feb/Mar/Apr. Non-December FY-end tickers contribute additional NaN months. Longest consecutive runs confirmed at 3--4 months.
- **`rev_q_revision_1m` (33.87% NaN):** NaN spikes to ~75% in months 2, 5, 8, 11 (off-months between quarterly earnings reports) because no quarterly estimate revision occurs between reporting windows.
- **`rev_q_dispersion` vs `rev_q_revision_1m` overlap:** investigated to confirm independent structural causes.

## Stage 4: Clean & Save

### Filtered to Universe (Stage 0)
Reduced from 1,150,491 rows (12,397 tickers) to 48,768 rows (231 tickers mapping to 227 PERMNOs), trimmed to 2004+.

### Columns Dropped (3)
- `rev_surprise`, `rev_surprise_pct` -- 99.44% NaN. Revenue surprise is only populated in the specific month when actuals are reported; only 274 valid rows out of 48,768. Unusable.
- `rev_q_revision_1m` -- 33.87% NaN. Structurally sparse: NaN spikes to ~75% in months 2, 5, 8, 11 (off-months between quarterly earnings reports). Above the 30% drop threshold.

### Dates Left as Mid-Month
Unlike price targets and recommendations (which use month-end dates), the revenue file uses IBES statistical period dates (e.g., Jan 15, Feb 19, Mar 18). These are correct and reflect when IBES actually computed the consensus. The merge pipeline will align all monthly data to trading days using `merge_asof(direction='backward')`, which handles different source dates naturally. Shifting dates here would lose point-in-time accuracy.

### No Winsorisation Applied Here
`rev_revision_3m` has extreme values (up to +3,591%) from fiscal year boundary effects. Winsorisation will be applied once, cross-sectionally per date, in the merge pipeline.

### Structural NaN Left as NaN
All NaN patterns have specific structural causes -- none are data quality issues:
- `rev_revision_1m`, `rev_numest_chg`, `rev_fy2_revision_1m`, `rev_eps_divergence` (~8.4% NaN each, ~4,100 rows): fiscal year rollover. 64.5% of NaN concentrates in February when most S&P 500 companies (December FY-end) roll from one fiscal year to the next. The 1-month revision becomes undefined because the prior month's estimate targeted a different fiscal year. ~21 NaN per ticker across 252 months = exactly 1 rollover per year.
- `rev_revision_3m` (25.2% NaN, 12,289 rows): same FY rollover extended. The 3-month lookback crosses a FY boundary for Feb/Mar/Apr, creating NaN in all 3 months after each December year-end. Tickers with non-December FY-ends contribute additional NaN months.
- `rev_dispersion` (1.05% NaN, 514 rows): 100% where `rev_numest = 1`. Single-analyst dispersion is undefined.
- `rev_q_dispersion` (2.5% NaN), `rev_fy2_dispersion` (1.2% NaN): same single-analyst pattern for quarterly and FY2 horizons.

None of these are forward-filled. The NaN is meaningful, and the cross-sectional aggregation in the merge pipeline will compute cap-weighted means from the ~92+ stocks with valid values each month.

## Output
`Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_revenue_clean.parquet` -- 12 factor columns (down from 15), 48,768 rows

In [1]:
# %% [markdown]
# # Data Cleaning: ibes_revenue.parquet
#
# Source: Data/Data_Collection/Initial/04_LSEG_IBES/ibes_revenue.parquet
# Output: Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_revenue_clean.parquet
#
# Monthly stock-level analyst revenue estimate data from WRDS IBES (statsum_xepsus).
# Like the other IBES files, the raw data contains ALL IBES tickers. We filter
# to our universe first using the clean link table.
#
# Expected factors (from collection code):
#   - rev_mean, rev_median, rev_high, rev_low: consensus revenue estimates ($M)
#   - rev_numest: number of analysts
#   - rev_dispersion: std/mean of revenue estimates
#   - rev_revision, rev_revision_3m: % change in mean estimate
#   - rev_surprise: actual vs estimate (after earnings release)

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH  = Path('../../../Data/Data_Collection/Initial/04_LSEG_IBES/ibes_revenue.parquet')
LINK_PATH = Path('../../../Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_permno_link_clean.parquet')
OUT_DIR   = Path('../../../Data/Data_Collection/Cleaned/04_LSEG_IBES')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
link = pd.read_parquet(LINK_PATH)

print(f"\n  Raw shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Raw tickers: {df['ticker'].nunique():,}")

# ── Filter to universe tickers ───────────────────────────────────────────────
valid_tickers = set(link['ticker'].unique())
n_before = len(df)
df = df[df['ticker'].isin(valid_tickers)].reset_index(drop=True)
print(f"\n  Filtered to universe tickers: {n_before:,} → {len(df):,} rows")
print(f"  Tickers retained: {df['ticker'].nunique()}")

# ── Trim to 2004-01-01 ──────────────────────────────────────────────────────
n_before = len(df)
df = df[df['date'] >= '2004-01-01'].reset_index(drop=True)
print(f"  Trimmed to 2004+: {n_before:,} → {len(df):,} rows")

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique()}")
print(f"  Unique tickers: {df['ticker'].nunique()}")
print(f"  Rows per ticker (mean): {df.groupby('ticker').size().mean():.1f}")
print(f"  Rows per ticker (median): {df.groupby('ticker').size().median():.0f}")

factor_cols = [c for c in df.columns if c not in ['ticker', 'date']]

print(f"\nColumns and dtypes ({len(factor_cols)} factors):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<25s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
print(df.head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df.tail(10).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN ───────────────────────────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<25s} {'NaN %':>8s}  {'Count':>8s}")
print("  " + "-" * 45)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    flag = " ← DROP" if pct >= 30 else (" ← INVESTIGATE" if pct >= 10 else "")
    print(f"  {col:<25s} {pct:>7.2f}%  {count:>8,d}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>8,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-3 NaN: {((row_nan >= 1) & (row_nan <= 3)).sum():>8,d}")
print(f"  Rows with 4-6 NaN: {((row_nan > 3) & (row_nan <= 6)).sum():>8,d}")
print(f"  Rows with >6 NaN: {(row_nan > 6).sum():>8,d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: VALUE RANGE & QUALITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: VALUE RANGE & QUALITY CHECKS")
print("=" * 90)

# ── 2a. Revenue estimate levels ──────────────────────────────────────────────
print(f"\n--- Revenue estimate levels ($M) ---")
for col in ['rev_mean', 'rev_median', 'rev_high', 'rev_low']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"  {col:<20s} min: ${vals.min():,.2f}  median: ${vals.median():,.2f}  "
          f"max: ${vals.max():,.2f}  mean: ${vals.mean():,.2f}")
    n_negative = (vals < 0).sum()
    if n_negative > 0:
        print(f"    ⚠ {n_negative} negative values")
    n_zero = (vals == 0).sum()
    if n_zero > 0:
        print(f"    ⚠ {n_zero} zero values")

# ── 2b. Analyst count ────────────────────────────────────────────────────────
numest_col = None
for candidate in ['rev_numest', 'rev_numrec', 'numest']:
    if candidate in df.columns:
        numest_col = candidate
        break

if numest_col:
    vals = df[numest_col].dropna()
    print(f"\n--- Analyst count ({numest_col}) ---")
    print(f"  range: {vals.min():.0f} – {vals.max():.0f}")
    print(f"  mean: {vals.mean():.1f}, median: {vals.median():.0f}")
    print(f"  Stocks with 1 analyst: {(vals == 1).sum():,} ({(vals == 1).mean()*100:.1f}%)")

# ── 2c. Revision columns ────────────────────────────────────────────────────
print(f"\n--- Revision columns ---")
for col in ['rev_revision', 'rev_revision_3m']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"  {col}:")
    print(f"    range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"    mean: {vals.mean():.4f}, median: {vals.median():.4f}")
    pctiles = vals.quantile([0.01, 0.05, 0.95, 0.99])
    print(f"    1st: {pctiles[0.01]:.4f}  5th: {pctiles[0.05]:.4f}  "
          f"95th: {pctiles[0.95]:.4f}  99th: {pctiles[0.99]:.4f}")
    n_extreme = ((vals > 100) | (vals < -50)).sum()
    if n_extreme > 0:
        print(f"    ⚠ {n_extreme} extreme values (>100% or <-50%)")

# ── 2d. Dispersion ──────────────────────────────────────────────────────────
if 'rev_dispersion' in df.columns:
    vals = df['rev_dispersion'].dropna()
    print(f"\n--- rev_dispersion ---")
    print(f"  range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"  mean: {vals.mean():.4f}, median: {vals.median():.4f}")
    pctiles = vals.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
    print(f"  1st: {pctiles[0.01]:.4f}  25th: {pctiles[0.25]:.4f}  "
          f"median: {pctiles[0.50]:.4f}  75th: {pctiles[0.75]:.4f}  "
          f"99th: {pctiles[0.99]:.4f}")

# ── 2e. Surprise ────────────────────────────────────────────────────────────
for col in ['rev_surprise', 'rev_surprise_pct']:
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    print(f"\n--- {col} ---")
    print(f"  range: {vals.min():.4f} – {vals.max():.4f}")
    print(f"  mean: {vals.mean():.4f}, median: {vals.median():.4f}")
    print(f"  NaN: {df[col].isna().sum()} ({df[col].isna().mean()*100:.2f}%)")
    pctiles = vals.quantile([0.01, 0.05, 0.95, 0.99])
    print(f"  1st: {pctiles[0.01]:.4f}  5th: {pctiles[0.05]:.4f}  "
          f"95th: {pctiles[0.95]:.4f}  99th: {pctiles[0.99]:.4f}")

# ── 2f. Consistency checks ──────────────────────────────────────────────────
print(f"\n--- Consistency checks ---")
if all(c in df.columns for c in ['rev_high', 'rev_low', 'rev_mean']):
    both_valid = df[['rev_high', 'rev_low', 'rev_mean']].dropna()
    n_inverted = (both_valid['rev_high'] < both_valid['rev_low']).sum()
    n_mean_outside = (
        (both_valid['rev_mean'] < both_valid['rev_low']) |
        (both_valid['rev_mean'] > both_valid['rev_high'])
    ).sum()
    print(f"  rev_high < rev_low (inverted): {n_inverted}")
    print(f"  rev_mean outside [rev_low, rev_high]: {n_mean_outside}")

# ── 2g. NaN pattern: dispersion when numest = 1 ─────────────────────────────
print(f"\n--- NaN pattern: dispersion when single analyst ---")
if numest_col and 'rev_dispersion' in df.columns:
    single_analyst = df[numest_col] == 1
    disp_nan = df['rev_dispersion'].isna()
    both = (single_analyst & disp_nan).sum()
    disp_nan_total = disp_nan.sum()
    print(f"  rev_dispersion NaN total: {disp_nan_total}")
    if disp_nan_total > 0:
        print(f"  Of those, where {numest_col} = 1: {both} ({both/disp_nan_total*100:.1f}%)")

# ── 2h. NaN pattern: revision in first month per ticker ─────────────────────
print(f"\n--- NaN pattern: revision in first month per ticker ---")
for col in ['rev_revision', 'rev_revision_3m']:
    if col not in df.columns:
        continue
    col_nan = df[col].isna()
    first_months = df.groupby('ticker')['date'].transform('min') == df['date']
    both = (first_months & col_nan).sum()
    nan_total = col_nan.sum()
    if nan_total > 0:
        print(f"  {col}: {nan_total} NaN, of those in first month(s): {both} ({both/nan_total*100:.1f}%)")
    else:
        print(f"  {col}: 0 NaN")

# ── 2i. Coverage per ticker ─────────────────────────────────────────────────
print(f"\n--- Coverage per ticker ---")
ticker_coverage = df.groupby('ticker').agg(
    n_months=('date', 'nunique'),
    first_date=('date', 'min'),
    last_date=('date', 'max')
)
print(f"  Months per ticker: mean={ticker_coverage['n_months'].mean():.0f}, "
      f"median={ticker_coverage['n_months'].median():.0f}, "
      f"min={ticker_coverage['n_months'].min()}, "
      f"max={ticker_coverage['n_months'].max()}")

short_coverage = ticker_coverage[ticker_coverage['n_months'] < 24]
if len(short_coverage) > 0:
    print(f"\n  Tickers with <24 months: {len(short_coverage)}")
    for ticker, row in short_coverage.iterrows():
        print(f"    {ticker:<10s} {row['n_months']:>3d} months  "
              f"({row['first_date'].date()} → {row['last_date'].date()})")

# ── 2j. Duplicate (ticker, date) ────────────────────────────────────────────
print(f"\n--- Duplicate (ticker, date) ---")
n_dupes = df.duplicated(subset=['ticker', 'date']).sum()
if n_dupes == 0:
    print(f"  ✓ No duplicates")
else:
    print(f"  ⚠ {n_dupes} duplicates")

# ── 2k. Date frequency ──────────────────────────────────────────────────────
print(f"\n--- Date frequency ---")
is_month_end = df['date'].dt.is_month_end
print(f"  Dates that are month-end: {is_month_end.sum():,} ({is_month_end.mean()*100:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: SUMMARY — DECISIONS NEEDED")
print("=" * 90)

print(f"""
Review the output above:

1. COLUMNS:
   - Any column with ≥30% NaN → drop
   - All others → keep (winsorisation and z-standardisation in merge pipeline)

2. NaN:
   - Structural NaN (dispersion with 1 analyst, revision in first month,
     surprise before earnings release) → leave as NaN
   - Any unexpected NaN patterns → investigate

3. VALUE RANGES:
   - Revenue estimates should be positive for most S&P 500 companies
   - Revisions: check percentiles for extreme outliers
   - Surprise: typically small (±5%) for large caps

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD, FILTER TO UNIVERSE, INSPECT

  Raw shape: 1,150,491 rows × 17 columns
  Raw tickers: 12,397

  Filtered to universe tickers: 1,150,491 → 51,122 rows
  Tickers retained: 229
  Trimmed to 2004+: 51,122 → 48,768 rows

  Shape: 48,768 rows × 17 columns
  Date range: 2004-01-15 → 2024-12-19
  Unique dates: 252
  Unique tickers: 229
  Rows per ticker (mean): 213.0
  Rows per ticker (median): 252

Columns and dtypes (15 factors):
    1. rev_mean_fy1              Float64        
    2. numest                    Float64        
    3. rev_dispersion            Float64        
    4. rev_range                 Float64        
    5. rev_revision_1m           Float64        
    6. rev_revision_3m           Float64        
    7. rev_numest                Float64        
    8. rev_numest_chg            Float64        
    9. rev_fy2_revision_1m       Float64        
   10. rev_fy2_dispersion        Float64        
   11. rev_q_revision_1m         Float64        
   12. rev_q_disper

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2b: DEEP NaN INVESTIGATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 2b: DEEP NaN INVESTIGATION")
print("=" * 90)

# ── Are the ~8.4% NaN columns the same rows? ────────────────────────────────
print(f"\n--- Are the ~8.4% NaN columns the same set of rows? ---")
cluster_cols = ['rev_revision_1m', 'rev_numest_chg', 'rev_fy2_revision_1m', 'rev_eps_divergence']
cluster_cols = [c for c in cluster_cols if c in df.columns]

if len(cluster_cols) >= 2:
    # Check pairwise overlap
    for i, col_a in enumerate(cluster_cols):
        for col_b in cluster_cols[i+1:]:
            nan_a = set(df[df[col_a].isna()].index)
            nan_b = set(df[df[col_b].isna()].index)
            overlap = len(nan_a & nan_b)
            print(f"  {col_a} ∩ {col_b}: {overlap} shared NaN rows "
                  f"(of {len(nan_a)} and {len(nan_b)})")

    # Are they ALL NaN on the same rows?
    all_nan_mask = df[cluster_cols].isna().all(axis=1)
    any_nan_mask = df[cluster_cols].isna().any(axis=1)
    print(f"\n  Rows where ALL 4 are NaN: {all_nan_mask.sum():,}")
    print(f"  Rows where ANY of 4 are NaN: {any_nan_mask.sum():,}")
    print(f"  → {'Same rows' if all_nan_mask.sum() == any_nan_mask.sum() else 'Different rows'}")

# ── What's special about the ~8.4% NaN rows? ────────────────────────────────
print(f"\n--- What characterises the ~8.4% NaN rows? ---")
ref_col = 'rev_revision_1m' if 'rev_revision_1m' in df.columns else cluster_cols[0]
nan_rows = df[df[ref_col].isna()].copy()
valid_rows = df[df[ref_col].notna()].copy()

print(f"  NaN rows: {len(nan_rows):,}")
print(f"  Valid rows: {len(valid_rows):,}")

# Is it first month per ticker?
first_month = df.groupby('ticker')['date'].transform('min') == df['date']
print(f"\n  NaN rows that are first month per ticker: "
      f"{(first_month & df[ref_col].isna()).sum()} "
      f"({(first_month & df[ref_col].isna()).sum() / len(nan_rows) * 100:.1f}%)")

# Is it a fiscal year rollover pattern? (certain months each year)
print(f"\n  NaN rows by month:")
nan_month_dist = nan_rows['date'].dt.month.value_counts().sort_index()
valid_month_dist = valid_rows['date'].dt.month.value_counts().sort_index()
for m in range(1, 13):
    n_nan = nan_month_dist.get(m, 0)
    n_valid = valid_month_dist.get(m, 0)
    total = n_nan + n_valid
    pct = n_nan / total * 100 if total > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"    Month {m:>2d}: {n_nan:>5d} NaN / {total:>5d} total ({pct:.1f}%) {bar}")

# Is it related to analyst count?
numest_col = 'rev_numest' if 'rev_numest' in df.columns else 'numest'
print(f"\n  Analyst count in NaN vs valid rows:")
print(f"    NaN rows:   mean={nan_rows[numest_col].mean():.1f}, "
      f"median={nan_rows[numest_col].median():.0f}")
print(f"    Valid rows: mean={valid_rows[numest_col].mean():.1f}, "
      f"median={valid_rows[numest_col].median():.0f}")

# Which tickers have the most NaN?
print(f"\n  Tickers with most NaN in {ref_col}:")
ticker_nan = nan_rows.groupby('ticker').size().sort_values(ascending=False)
for ticker, n in ticker_nan.head(15).items():
    total = len(df[df['ticker'] == ticker])
    print(f"    {ticker:<10s} {n:>4d} NaN / {total:>4d} total ({n/total*100:.1f}%)")

# ── rev_revision_3m: why 25% NaN? ───────────────────────────────────────────
print(f"\n--- rev_revision_3m: investigating 25% NaN ---")
if 'rev_revision_3m' in df.columns:
    rev3_nan = df[df['rev_revision_3m'].isna()].copy()

    # Is it first 3 months per ticker?
    df['_month_rank'] = df.groupby('ticker')['date'].rank(method='first')
    first_3 = df['_month_rank'] <= 3
    rev3_nan_mask = df['rev_revision_3m'].isna()
    print(f"  Total NaN: {rev3_nan_mask.sum():,}")
    print(f"  In first 3 months per ticker: {(first_3 & rev3_nan_mask).sum()}")
    print(f"  NOT in first 3 months: {(~first_3 & rev3_nan_mask).sum()}")

    # For the non-first-3 NaN, what's the pattern?
    non_first3_nan = df[~first_3 & rev3_nan_mask]
    print(f"\n  Non-first-3 NaN rows by month:")
    month_dist = non_first3_nan['date'].dt.month.value_counts().sort_index()
    for m in range(1, 13):
        n = month_dist.get(m, 0)
        print(f"    Month {m:>2d}: {n:>5d}")

    # Do they overlap with the ~8.4% cluster?
    cluster_nan = df[cluster_cols].isna().all(axis=1)
    both = (cluster_nan & rev3_nan_mask & ~first_3).sum()
    print(f"\n  Non-first-3 rev_revision_3m NaN that are ALSO in the 8.4% cluster: "
          f"{both} ({both / max((~first_3 & rev3_nan_mask).sum(), 1) * 100:.1f}%)")

    # Consecutive NaN runs per ticker
    print(f"\n  Longest consecutive rev_revision_3m NaN runs per ticker (top 10):")
    for ticker in df['ticker'].unique():
        sub = df[df['ticker'] == ticker].sort_values('date')
        is_nan = sub['rev_revision_3m'].isna()
        if not is_nan.any():
            continue
        runs = is_nan.ne(is_nan.shift()).cumsum()
        nan_runs = sub[is_nan].groupby(runs).size()
        if nan_runs.max() > 3:
            longest = nan_runs.max()
            print(f"    {ticker:<10s} longest run: {longest} months")

    df = df.drop(columns='_month_rank')

# ── rev_q_revision_1m: why 34% NaN? ─────────────────────────────────────────
print(f"\n--- rev_q_revision_1m: investigating 34% NaN ---")
if 'rev_q_revision_1m' in df.columns:
    q_nan = df[df['rev_q_revision_1m'].isna()].copy()

    # Is it a quarterly pattern? (NaN in months between quarterly reports)
    print(f"  NaN rows by month:")
    month_dist = q_nan['date'].dt.month.value_counts().sort_index()
    total_by_month = df['date'].dt.month.value_counts().sort_index()
    for m in range(1, 13):
        n_nan = month_dist.get(m, 0)
        n_total = total_by_month.get(m, 0)
        pct = n_nan / n_total * 100 if n_total > 0 else 0
        bar = "█" * int(pct / 2)
        print(f"    Month {m:>2d}: {n_nan:>5d} NaN / {n_total:>5d} total ({pct:.1f}%) {bar}")

    # Per-ticker: how many NaN?
    ticker_nan_pct = q_nan.groupby('ticker').size() / df.groupby('ticker').size() * 100
    print(f"\n  Per-ticker NaN rate:")
    print(f"    mean: {ticker_nan_pct.mean():.1f}%")
    print(f"    median: {ticker_nan_pct.median():.1f}%")
    print(f"    Tickers with >50% NaN: {(ticker_nan_pct > 50).sum()}")

# ── rev_q_dispersion vs rev_q_revision_1m NaN overlap ───────────────────────
print(f"\n--- rev_q_dispersion vs rev_q_revision_1m NaN overlap ---")
if all(c in df.columns for c in ['rev_q_dispersion', 'rev_q_revision_1m']):
    q_disp_nan = df['rev_q_dispersion'].isna().sum()
    q_rev_nan = df['rev_q_revision_1m'].isna().sum()
    both = (df['rev_q_dispersion'].isna() & df['rev_q_revision_1m'].isna()).sum()
    print(f"  rev_q_dispersion NaN: {q_disp_nan}")
    print(f"  rev_q_revision_1m NaN: {q_rev_nan}")
    print(f"  Both NaN: {both}")
    print(f"  rev_q_revision_1m NaN but rev_q_dispersion valid: {q_rev_nan - both}")

STAGE 2b: DEEP NaN INVESTIGATION

--- Are the ~8.4% NaN columns the same set of rows? ---
  rev_revision_1m ∩ rev_numest_chg: 4103 shared NaN rows (of 4103 and 4103)
  rev_revision_1m ∩ rev_fy2_revision_1m: 4103 shared NaN rows (of 4103 and 4153)
  rev_revision_1m ∩ rev_eps_divergence: 4103 shared NaN rows (of 4103 and 4119)
  rev_numest_chg ∩ rev_fy2_revision_1m: 4103 shared NaN rows (of 4103 and 4153)
  rev_numest_chg ∩ rev_eps_divergence: 4103 shared NaN rows (of 4103 and 4119)
  rev_fy2_revision_1m ∩ rev_eps_divergence: 4103 shared NaN rows (of 4153 and 4119)

  Rows where ALL 4 are NaN: 4,103
  Rows where ANY of 4 are NaN: 4,169
  → Different rows

--- What characterises the ~8.4% NaN rows? ---
  NaN rows: 4,103
  Valid rows: 44,665

  NaN rows that are first month per ticker: 42 (1.0%)

  NaN rows by month:
    Month  1:   316 NaN /  4068 total (7.8%) ███
    Month  2:  2623 NaN /  4065 total (64.5%) ████████████████████████████████
    Month  3:   359 NaN /  4065 total (8.8%) ██

In [3]:
# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Filtered to universe (Stage 0):**
# The raw file contains 1,150,491 rows across 12,397 IBES tickers. Filtered
# to the 231 tickers mapping to our 227 top-100 S&P 500 PERMNOs using the
# clean link table, then trimmed to 2004+. This reduced the data from
# 1.15M to 48,768 rows.
#
# **Columns dropped (3):**
# - `rev_surprise`, `rev_surprise_pct` — 99.44% NaN. Revenue surprise is only
#   populated in the specific month when actuals are reported, and even then
#   most observations are missing. Only 274 valid rows out of 48,768. Unusable.
# - `rev_q_revision_1m` — 33.87% NaN. The quarterly revision is structurally
#   sparse: NaN spikes to ~75% in months 2, 5, 8, 11 (the off-months between
#   quarterly earnings reports) because no quarterly estimate revision occurs
#   between reporting windows. Above the 30% drop threshold.
#
# **Dates left as mid-month (not shifted to month-end):**
# Unlike price targets and recommendations (which use month-end dates), the
# revenue file uses IBES statistical period dates — mid-month snapshots like
# January 15, February 19, March 18. These are correct and reflect when IBES
# actually computed the consensus. The merge pipeline will align all monthly
# data to trading days using `merge_asof(direction='backward')`, which handles
# different source dates naturally. Shifting dates here would lose point-in-time
# accuracy.
#
# **No winsorisation applied here.** `rev_revision_3m` has extreme values (up
# to +3,591%) from fiscal year boundary effects. Winsorisation will be applied
# once, cross-sectionally per date, in the merge pipeline.
#
# **Structural NaN left as NaN:**
# All NaN patterns have specific structural causes:
# - `rev_revision_1m`, `rev_numest_chg`, `rev_fy2_revision_1m`,
#   `rev_eps_divergence` (~8.4% NaN each, ~4,100 rows): fiscal year rollover.
#   64.5% of the NaN concentrates in February when most S&P 500 companies
#   (December FY-end) roll from FY1=2023 to FY1=2024. The 1-month revision
#   becomes undefined because last month's estimate targeted a different fiscal
#   year. ~21 NaN per ticker across 252 months = exactly 1 rollover per year.
# - `rev_revision_3m` (25.2% NaN, 12,289 rows): same FY rollover extended.
#   The 3-month lookback crosses a FY boundary for Feb/Mar/Apr, creating NaN
#   in all 3 months after each December year-end. Tickers with non-December
#   FY-ends (e.g., June) contribute additional NaN months.
# - `rev_dispersion` (1.05% NaN, 514 rows): 100% where `rev_numest = 1`.
#   Single-analyst dispersion is undefined.
# - `rev_q_dispersion` (2.5% NaN), `rev_fy2_dispersion` (1.2% NaN): same
#   single-analyst pattern for quarterly and FY2 horizons.
#
# None of these are forward-filled. The NaN is meaningful (not missing), and
# the cross-sectional aggregation in the merge pipeline will compute cap-weighted
# means from the ~92+ stocks that do have valid values each month.
#
# **Factors retained: 12** (was 15 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

# ── 4a. Drop columns ────────────────────────────────────────────────────────
drop_cols = ['rev_surprise', 'rev_surprise_pct', 'rev_q_revision_1m']
drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)

factor_cols = [c for c in df.columns if c not in ['ticker', 'date']]
print(f"\n  Dropped {len(drop_cols_present)} columns: {drop_cols_present}")
print(f"  Remaining: {len(factor_cols)} factor columns")

# ── 4b. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols].isna().sum()
nan_cols = nan_check[nan_check > 0]
if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN")
else:
    total_nan = nan_cols.sum()
    print(f"\n  Remaining NaN: {total_nan:,} (structural, left intentionally)")
    for col, n in nan_cols.items():
        pct = n / len(df) * 100
        print(f"    {col:<25s} {n:>6,d} NaN ({pct:.1f}%)")

# ── 4c. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Tickers: {df['ticker'].nunique()}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    vals = df[c].dropna()
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n:,} NaN, {nan_n/len(df)*100:.1f}%)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<25s} range: [{vals.min():.4f}, {vals.max():.4f}]{nan_str}")

print(f"\n  Sample (first 5 rows):")
print(df.head(5).to_string(index=False))

# ── 4d. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'ibes_revenue_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 4: CLEAN & SAVE

  Dropped 3 columns: ['rev_surprise', 'rev_surprise_pct', 'rev_q_revision_1m']
  Remaining: 12 factor columns

  Remaining NaN: 31,078 (structural, left intentionally)
    rev_dispersion               514 NaN (1.1%)
    rev_revision_1m            4,103 NaN (8.4%)
    rev_revision_3m           12,289 NaN (25.2%)
    rev_numest_chg             4,103 NaN (8.4%)
    rev_fy2_revision_1m        4,153 NaN (8.5%)
    rev_fy2_dispersion           580 NaN (1.2%)
    rev_q_dispersion           1,217 NaN (2.5%)
    rev_eps_divergence         4,119 NaN (8.4%)

  Final shape: 48,768 rows × 14 columns
  Tickers: 229
  Date range: 2004-01-15 → 2024-12-19

  Factor list (12 columns):
      1. rev_mean_fy1              range: [-13278.8000, 710453.6500]
      2. numest                    range: [1.0000, 64.0000]
      3. rev_dispersion            range: [0.0001, 11.0929]  (514 NaN, 1.1%)
      4. rev_range                 range: [0.0000, 28.8260]
      5. rev_revision_1m           